<a href="https://colab.research.google.com/github/MelvTheGoat/Machine_Learning/blob/main/Text_Summarization_using_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import numpy as np
import pandas as pd
import pickle
from statistics import mode
import nltk
from nltk import word_tokenize
from nltk.stem import LancasterStemmer
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from tensorflow.keras.models import Model
from tensorflow.keras import models
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import plot_model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Concatenate, Attention
from sklearn.model_selection import train_test_split
from bs4 import BeautifulSoup

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
import kagglehub

path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'amazon-fine-food-reviews' dataset.
Path to dataset files: /kaggle/input/amazon-fine-food-reviews


In [ ]:
df = pd.read_csv(path + "/Reviews.csv", nrows=100000)
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [ ]:
df.drop_duplicates(subset=['Text'], inplace=True)
df.dropna(axis=0, inplace=True)
input_text = df.loc[:, 'Text']
target_text = df.loc[:, 'Summary']
target_text.replace('', np.nan, inplace=True)

In [ ]:
# Defining a standard contractions dictionary manually since the .pkl file is missing
contractions_dict = {
    "ain't": "is not", "aren't": "are not","can't": "cannot", "'cause": "because", "could've": "could have",
    "couldn't": "could not", "didn't": "did not",  "doesn't": "does not", "don't": "do not", "hadn't": "had not",
    "hasn't": "has not", "haven't": "have not", "he'd": "he would","he'll": "he will", "he's": "he is",
    "how'd": "how did", "how'd'y": "how do you", "how'll": "how will", "how's": "how is",
    "I'd": "I would", "I'd've": "I would have", "I'll": "I will", "I'll've": "I will have","I'm": "I am",
    "I've": "I have", "i'd": "i would", "i'd've": "i would have", "i'll": "i will",  "i'll've": "i will have",
    "i'm": "i am", "i've": "i have", "isn't": "is not", "it'd": "it would", "it'd've": "it would have",
    "it'll": "it will", "it'll've": "it will have","it's": "it is", "let's": "let us", "ma'am": "madam",
    "mayn't": "may not", "might've": "might have","mightn't": "might not","mightn't've": "might not have",
    "must've": "must have", "mustn't": "must not", "mustn't've": "must not have", "needn't": "need not",
    "needn't've": "need not have","o'clock": "of the clock", "oughtn't": "ought not", "oughtn't've": "ought not have",
    "shan't": "shall not", "sha'n't": "shall not", "shan't've": "shall not have", "she'd": "she would",
    "she'd've": "she would have", "she'll": "she will", "she'll've": "she will have", "she's": "she is",
    "should've": "should have", "shouldn't": "should not", "shouldn't've": "should not have", "so've": "so have",
    "so's": "so as", "this's": "this is","that'd": "that would", "that'd've": "that would have", "that's": "that is",
    "there'd": "there would", "there'd've": "there would have", "there's": "there is", "here's": "here is",
    "they'd": "they would", "they'd've": "they would have", "they'll": "they will", "they'll've": "they will have",
    "they're": "they are", "they've": "they have", "to've": "to have", "wasn't": "was not", "we'd": "we would",
    "we'd've": "we would have", "we'll": "we will", "we'll've": "we will have", "we're": "we are",
    "we've": "we have", "weren't": "were not", "what'll": "what will", "what'll've": "what will have",
    "what're": "what are",  "what's": "what is", "what've": "what have", "when's": "when is",
    "when've": "when have", "where'd": "where did", "where's": "where is", "where've": "where have",
    "who'll": "who will", "who'll've": "who will have", "who's": "who is", "who've": "who have",
    "why's": "why is", "why've": "why have", "will've": "will have", "won't": "will not", "won't've": "will not have",
    "would've": "would have", "wouldn't": "would not", "wouldn't've": "would not have", "y'all": "you all",
    "y'all'd": "you all would","y'all'd've": "you all would have","y'all're": "you all are","y'all've": "you all have",
    "you'd": "you would", "you'd've": "you would have", "you'll": "you will", "you'll've": "you will have",
    "you're": "you are", "you've": "you have"
}

In [ ]:
# Updated processing cell to use the defined dictionary
input_text = []
target_text = []
input_words = []
target_words = []

# Using the local dictionary instead of pickle.load
contractions = contractions_dict

stop_words = set(stopwords.words('english'))
stemm = LancasterStemmer()

In [ ]:
def clean(texts, src):
    texts = BeautifulSoup(texts, "lxml").text
    words = word_tokenize(texts.lower())
    words = list(filter(lambda w: (w.isalpha() and len(w)>=3), words))

In [ ]:
def clean(texts, src):
    texts = BeautifulSoup(texts, "lxml").text
    words = word_tokenize(texts.lower())
    words = list(filter(lambda w: (w.isalpha() and len(w)>=3), words))

    words = [contractions[w] if w in contractions else w for w in words]
    if src == "inputs":
        words = [stemm.stem(w) for w in words if w not in stop_words]
    else:
        words = [w for w in words if w not in stop_words]
    return words

In [ ]:
input_texts = []
target_texts = []
input_words = []
target_words = []

for in_txt, tr_txt in zip(df['Text'], df['Summary']):
    in_words = clean(in_txt, "inputs")
    input_texts.append(' '.join(in_words))
    input_words.extend(in_words)

    tr_words = clean("sos " + tr_txt  + " eos", "target")
    target_texts.append(' '.join(tr_words))
    target_words.extend(tr_words)

input_words = sorted(list(set(input_words)))
target_words = sorted(list(set(target_words)))
num_in_words = len(input_words)
num_tr_words = len(target_words)

max_in_len = max([len(i.split()) for i in input_texts]) # Use max instead of mode for length
max_tr_len = max([len(i.split()) for i in target_texts]) # Use max instead of mode for length

print("Number of input words:", num_in_words)
print("Number of target words:", num_tr_words)
print("Max input length:", max_in_len)
print("Max target length:", max_tr_len)

X_train, X_test, y_train, y_test = train_test_split(input_texts, target_texts, test_size=0.2, random_state=42)

in_tokenizer = Tokenizer()
in_tokenizer.fit_on_texts(X_train)
tr_tokenizer = Tokenizer()
tr_tokenizer.fit_on_texts(y_train)

X_train = in_tokenizer.texts_to_sequences(X_train)
y_train = tr_tokenizer.texts_to_sequences(y_train)

en_in_data = pad_sequences(X_train, maxlen=max_in_len, padding='post')
dec_data = pad_sequences(y_train, maxlen=max_tr_len, padding='post')

dec_in_data = dec_data[:, :-1]
dec_tr_data = dec_data.reshape(len(dec_data), max_tr_len, 1)[:, 1:]

K.clear_session()
latent_dim = 500

en_inputs = Input(shape=(max_in_len,))
en_embedding = Embedding(num_in_words + 1, latent_dim)(en_inputs)

# LSTM 1
en_lstm1 = LSTM(latent_dim, return_state=True, return_sequences=True)
en_outputs1, state_h1, state_c1 = en_lstm1(en_embedding)

# LSTM 2
en_lstm2 = LSTM(latent_dim, return_state=True, return_sequences=True)
en_outputs2, state_h2, state_c2 = en_lstm2(en_outputs1)

# LSTM 3
en_lstm3 = LSTM(latent_dim, return_state=True, return_sequences=True)
en_outputs3, state_h3, state_c3 = en_lstm3(en_outputs2)


# encoder states
en_states = [state_h3, state_c3]

dec_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(num_tr_words + 1, latent_dim)
dec_embedding = dec_emb_layer(dec_inputs)
dec_lstm = LSTM(latent_dim, return_state=True, return_sequences=True)
dec_outputs, _, _ = dec_lstm(dec_embedding, initial_state=en_states)

attention = Attention()
attn_out = attention([dec_outputs, en_outputs3])
merge = Concatenate(axis=-1, name='concat_layer1')([dec_outputs, attn_out])

dec_dense = Dense(num_tr_words + 1, activation='softmax')
dec_outputs = dec_dense(merge)

model = Model([en_inputs, dec_inputs], dec_outputs)
model.summary()
plot_model(model, to_file='model.png', show_shapes=True, show_layer_names=True)

model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=["accuracy"])
model.fit([en_in_data, dec_in_data], dec_tr_data, batch_size=512, epochs=50, validation_split=0.1)
model.save("model.h5")

# Inference model setup (these should not be inside the data processing loop either)
latenth_dim = 500 # This variable was redefined, but let's assume it should be latent_dim
# The next line uses model.h5 which was supposed to be saved above. If the training completes, this should work.
model = models.load_model("model.h5")

en_outputs, state_h_enc, state_c_enc = model.layers[6].output
en_states = [state_h_enc, state_c_enc]
en_model = Model(model.input[0], [en_outputs] + en_states)

dec_state_input_h = Input(shape=(latent_dim,))
dec_state_input_c = Input(shape=(latent_dim,))
dec_states_inputs = [dec_state_input_h, dec_state_input_c]

dec_inputs = model.input[1]
dec_emb_layer = model.layers[5]
dec_lstm = model.layers[7]
dec_embedding = dec_emb_layer(dec_inputs)

dec_outputs2, state_h2, state_c2 = dec_lstm(dec_embedding, initial_state=dec_states_inputs)

attention = model.layers[8]
attn_out2 = attention([dec_outputs2, en_outputs])
merge2 = Concatenate(axis=-1, name='concat_layer2')([dec_outputs2, attn_out2])

dec_dense = model.layers[10]
dec_outputs2 = dec_dense(merge2)

dec_model = Model([dec_inputs] + dec_states_inputs + [en_outputs], [dec_outputs2] + [state_h2, state_c2])

reverse_target_word_index = tr_tokenizer.index_word
reverse_source_word_index = in_tokenizer.index_word
target_word_index = tr_tokenizer.word_index
reverse_target_word_index[0] = ''

def decode_sequence(input_seq):
    en_out, en_h, en_c = en_model.predict(input_seq)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_word_index['sos']

    stop_condition = False
    decoded_sentence = ''
    while not stop_condition:
        output_tokens, h, c = dec_model.predict([target_seq] + [en_h, en_c] + [en_out])

        word_index = np.argmax(output_tokens[0, -1, :])
        text_word = reverse_target_word_index[word_index]
        decoded_sentence += text_word + ' '
        if text_word == 'eos' or len(decoded_sentence.split()) >= max_tr_len:
            stop_condition = True
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = word_index
        en_h, en_c = h, c
    return decoded_sentence

# User input and prediction part (removed indentation to run once after model setup)
# inp_review = input("Enter a review: ")
# print("Review :", inp_review)
# inp_review = clean(inp_review, "inputs")
# inp_review = ' '.join(inp_review)
# inp_x = in_tokenizer.texts_to_sequences([inp_review])
# inp_x = pad_sequences(inp_x, maxlen=max_in_len, padding='post')
# summary = decode_sequence(inp_x.reshape(1, max_in_len))
# print("Predicted summary:", summary)

Number of input words: 32198
Number of target words: 14171
Max input length: 1365
Max target length: 18


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1365)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1365, 500) │ 16,099,500 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 1365,     │  2,002,000 │ embedding[0][0]   │
│                     │ 500), (None,      │            │                   │
│                     │ 500), (None,      │            │                   │
│                     │ 500)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 1365,     │  2,002,000 │ lstm[0][0]        │
│                     │ 500), (None,      │            │                   │
│                     │ 500), (None,      │            │                   │
│                     │ 500)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 500) │  7,086,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 1365,     │  2,002,000 │ lstm_1[0][0]      │
│                     │ 500), (None,      │            │                   │
│                     │ 500), (None,      │            │                   │
│                     │ 500)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, None,     │  2,002,000 │ embedding_1[0][0… │
│                     │ 500), (None,      │            │ lstm_2[0][1],     │
│                     │ 500), (None,      │            │ lstm_2[0][2]      │
│                     │ 500)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, None, 500) │          0 │ lstm_3[0][0],     │
│ (Attention)         │                   │            │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_layer1       │ (None, None,      │          0 │ lstm_3[0][0],     │
│ (Concatenate)       │ 1000)             │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │ 14,186,172 │ concat_layer1[0]… │
│                     │ 14172)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 45,379,672 (173.11 MB)

 Trainable params: 45,379,672 (173.11 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
